In [0]:
-- =============================================================================
-- SECTION 3: CONTENT VIEW
-- =============================================================================

-- CV-1  Top 20 pages by edit count — last hour
-- Widget: Table  |  sort by total_edits desc
-- ─────────────────────────────────────────────────────────────────────────────
SELECT
  title,
  CASE namespace
    WHEN 0 THEN 'article'
    WHEN 1 THEN 'talk'
    WHEN 4 THEN 'project'
    WHEN 10 THEN 'template'
    ELSE CAST(namespace AS STRING)
  END                         AS namespace,
  SUM(edit_count)             AS total_edits,
  SUM(unique_editors)         AS editors,
  SUM(total_byte_delta)       AS net_bytes,
  SUM(minor_edit_count)       AS minor_edits
FROM wiki_poc.poc.gold_page_edits_5min
WHERE window_start >= NOW() - INTERVAL 1 HOUR
GROUP BY title, namespace
ORDER BY total_edits DESC
LIMIT 20;


-- CV-2  Overall edit volume over time — last 6 hours
-- Widget: Line chart  |  x=window_start  y=total_edits  y2=pages_edited
-- ─────────────────────────────────────────────────────────────────────────────
SELECT
  window_start,
  SUM(edit_count)          AS total_edits,
  COUNT(DISTINCT title)    AS pages_edited,
  SUM(total_byte_delta)    AS net_bytes_added
FROM wiki_poc.poc.gold_page_edits_5min
WHERE window_start >= NOW() - INTERVAL 6 HOURS
GROUP BY window_start
ORDER BY window_start;


-- CV-3  Largest single edits — last hour (by absolute byte delta)
-- Widget: Table
-- Useful for spotting mass-reverts (large negative delta) or major additions
-- ─────────────────────────────────────────────────────────────────────────────
SELECT
  title,
  user,
  event_timestamp,
  byte_delta,
  SUBSTRING(comment, 1, 120)  AS comment_preview
FROM wiki_poc.poc.silver_recentchange_enwiki
WHERE event_timestamp >= NOW() - INTERVAL 1 HOUR
  AND byte_delta IS NOT NULL
ORDER BY ABS(byte_delta) DESC
LIMIT 10;